In [1]:
#pip install ollama
import os
from ollama import Client
import pandas as pd
client = Client(
  host='https://ollama.com',
  headers={'Authorization': '37708851ece74a35af16b6a3ef745f0d.qlVFnGl6jJE7R5ULW3BF9tKe'}
)

response = client.chat(model='gpt-oss:20b', messages=[
  {'role': 'user', 'content': 'Test?'},
])
print(response['message']['content'])

Hello! It looks like you’re just checking in. How can I help you today?


Importing ollama as llm to check a few companies and adjust the message

In [2]:
def query(question):
    system_instruction = """
You are an instrument which will help me understand a few thing from a query.
You will receive a query and you will return a few things in the next format:
Public: 0/1/2(false,true,unknown) - a number
Revenue: 0/1/2(under/over),2 if the querry does not contain this information,value - a number
Employees:0/1/2(under/over),2 if the querry does not contain this information,value - a number
Year:0/1/2(under/over),2 if the querry does not contain this information,value - a number
Description: string, everything else that is not included in the above categories...ELIMINA TOT CE AI FOLOSIT LA INTREBARILE PRECEDENTE
Example:
Query: Public software companies with more than 1,000 employees.
Answer:
Public: 1
Revenue: 2
Employees: 1,1000
Year: 2
Description: software companies

Example 2:
Query: Fast-growing fintech companies competing with traditional banks in Europe.
Answer:
Public: 2
Revenue: 2
Employees: 2
Year: 2
Description: Fast-growing fintech companies competing with traditional banks in Europe

Nimic in plus,nu json,nu markdown,nu text,doar raspunsul in formatul de mai sus., int si string,nu alte tipuri de date.
"""

    response = client.chat(model='gpt-oss:20b', messages=[
        {'role': 'system', 'content': system_instruction},
        {'role': 'user', 'content': question},
    ],
    options={
        'temperature': 0      
    })
    return(response['message']['content'])
query("Pharmaceutical companies in Switzerland")


'Public: 2\nRevenue: 2\nEmployees: 2\nYear: 2\nDescription: Pharmaceutical companies in Switzerland'

In [ ]:
import joblib
%pip install --upgrade scikit-learn
tfidf_matrix = joblib.load(r'C:\\ML-project\\matrix_tfidf.joblib')
kmeans_model = joblib.load(r'C:\\ML-project\\model_kmeans_10.joblib')
tfidf = joblib.load(r'C:\\ML-project\\vectorizer_tfidf.joblib')


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [4]:
def search_cluster(description):
    vector_text = tfidf.transform([description])
    cluster_identificat = kmeans_model.predict(vector_text)[0]
    return cluster_identificat

description_test = "Suppliers of software solutions"
rezultat = search_cluster(description_test)
print(rezultat)


6


Now for every query we take the descrition and assign it to the specific cluster

In [13]:
def get_candidates(query_response_text, cluster_id):
    file_path = f'C:\\ML-project\\cluster_{cluster_id}.json'
    
    try:
        df = pd.read_json(file_path)
    except FileNotFoundError:
        return None
    filters = {}
    for line in query_response_text.strip().split('\n'):
        if ':' in line:
            k, v = line.split(':', 1)
            filters[k.strip()] = v.strip()

    if filters.get('Public') == '1':
        df = df[df['is_public'] == True]
    elif filters.get('Public') == '0':
        df = df[df['is_public'] == False]

    if 'Employees' in filters and filters['Employees'] != '2':
        try:
            mod, val = filters['Employees'].split(',')
            val = float(val)
            if mod == '1': 
                df = df[df['employees_count'] > val]
            elif mod == '0': 
                df = df[df['employees_count'] < val]
        except:
            pass

    if 'Revenue' in filters and filters['Revenue'] != '2':
        try:
            mod, val = filters['Revenue'].split(',')
            val = float(val)
            if mod == '1': 
                df = df[df['revenue'] > val]
            elif mod == '0': 
                df = df[df['revenue'] < val]
        except:
            pass

    if 'Year' in filters and filters['Year'] != '2':
        try:
            mod, val = filters['Year'].split(',')
            val = float(val)
            if mod == '1':
                df = df[df['year_founded'] > val]
            elif mod == '0':
                df = df[df['year_founded'] < val]
        except:
            pass

    return df

Now we fillter one more time according to the revenue,year,public,and no of employees within that cluster.

In [14]:
def top_solutions(user_query):
    system_instruction = """You are an instrument which will help me understand a few thing from a query.
You will receive a query and a company data and you will return a score between 0 to 1 based on how well the company data matches the query:1 - very good match, 0 - no match at all. Focus on the description of the company and how well it matches the query. Do not take into account the other categories, only the description. Do not return anything else.JUST A NUMBER FLOAT.
Verify the location if it is mentioned and all the information. 0.5 means not bad but not the best match.Be more harsh in your scoring, a company with 1 should be perfect, I want to select like best 5, I don't need more 1's than 3,4"""

    filters = query(user_query)
    parsed_filters = {}
    for line in filters.strip().split('\n'):
        if ':' in line:
            k, v = line.split(':', 1)
            parsed_filters[k.strip()] = v.strip()
    
    description_for_cluster = parsed_filters.get("Description", user_query).strip()
    cluster_id = search_cluster(description_for_cluster)
    candidates = get_candidates(filters, cluster_id)
    if candidates is None or (isinstance(candidates, pd.DataFrame) and candidates.empty):
        return "Companies not found."

    scores = []
    for _, row in candidates.iterrows():
        company_data = row.to_json() 
        company_info = f"Full Company Data: {company_data}"
        
        try:
            score_response = client.chat(
                model='gpt-oss:20b', 
                messages=[
                    {'role': 'system', 'content': system_instruction},
                    {'role': 'user', 'content': f"Query: {user_query}\nCompany Data: {company_info}"}
                ], 
                options={'temperature': 0}
            )
            
            score_text = score_response['message']['content'].strip()
            score = float(score_text)
            scores.append(score)
        except:
            scores.append(0.0)

    candidates = candidates.copy()
    candidates['relevance_score'] = scores
    final_top = candidates.sort_values(by='relevance_score', ascending=False)
    
    return(final_top)


    

In [32]:
def display_top_results(user_query):
    results = top_solutions(user_query)
    if results is None or results.empty:
        with open('C:\\ML-project\\Qualification-Results.txt', 'a') as f:
            f.write(user_query + "\n\n")
            f.write("Companies not found.\n\n")
        return
    filtered_df = results[results['relevance_score'] > 0.3].copy()
    
    if filtered_df.empty:
        with open('C:\\ML-project\\Qualification-Results.txt', 'a') as f:
            f.write(user_query + "\n\n")
            f.write("Not good enough companies found.\n\n")
        return

    top_5 = filtered_df.head(5)
    if 'address' in top_5.columns:
        top_5['country_code'] = top_5['address'].apply(
            lambda x: x.get('country_code') if isinstance(x, dict) else "N/A"
        )
    else:
        top_5['country_code'] = "N/A"

    final_output = top_5[['operational_name', 'website', 'country_code', 'description']]
    
    with open('C:\\ML-project\\Qualification-Results.txt', 'a',encoding='utf-8') as f:
        f.write(user_query + "\n\n")
        for _, row in final_output.iterrows():
            f.write(f"Name: {row['operational_name']}\n")
            f.write(f"Website: {row['website']}\n")
            f.write(f"Country Code: {row['country_code']}\n")
            f.write(f"Description: {row['description']}\n")
            f.write("\n")
        f.write("\n\n")
display_top_results("Companies that manufacture or supply critical components for electric vehicle battery production")
    

C:\Users\petru\AppData\Local\Temp\ipykernel_17768\1149766397.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_5['country_code'] = top_5['address'].apply(
